# Parameter setzen:

In [ ]:
import os
import numpy as np
import sys
import scipy.io as sio
from pathlib import Path
%matplotlib widget

# GPU wählen
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

SNR = 7

# Run-Config
model = f"SNR{SNR}"
GT_path = f"/workspace/Denoising/datasets/DMI/Simulations/Lesion_double_radius/Simulated_Lesion_GT_double_normalized/data.npy"

batch_size = 600
Vol = "Vol01_WB/Res64x64_Thick"#_Brisbane"
multi_view_mode = "single"  # options: "single" (default), "stack", "average"
add_water = False # adde water back grom hankel decomposition

#Scanner = "3T"

# 🔹 NEU: Ordner + Dateiname getrennt

data_dir = Path(f"/workspace/Denoising/datasets/DMI/Simulations/SNRExperiments_double_radius/SNR{SNR}/Noise_6")
#data_dir = Path(f"../datasets/Proton/{Scanner}/{Vol}/OriginalData")
file_name = "data_tMPPCA.npy" #data_after_walinet

#magnitude = np.load(f"../datasets/Proton/{Scanner}/{Vol}/OriginalData/magnitude.npy")

model_input = data_dir / file_name



# ---- Comparisons definieren ----
comparison_paths = [
    model_input,
    GT_path,
]

Titles = [
    "Deep",
    "No Denoising",
    "GT",
]

# ---- Daten laden (einmalig!) ----
Data = []

for p in comparison_paths:
    p = Path(p)

    if p.suffix == ".npy":
        arr = np.load(p)

    elif p.suffix == ".mat":
        mat = loadmat(p)
        arr = np.asarray(mat["csi"]["Data"][0,0])

    else:
        raise ValueError(f"Unsupported file format: {p}")

    Data.append(arr)

# Inference

In [ ]:
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from denoising.config.load import load_yaml
from denoising.config.build import build_config
from denoising.inference.api import infer
from src.denoising.figures.EvalBeforeFitting import *

config_path = f"../trained_models/{model}/train.yaml"

cfg = build_config(load_yaml(config_path))

denoised, meta = infer(
    cfg=cfg,
    ckpt_path=f"../trained_models/{model}/checkpoints/last.pt",
    input_path=model_input,
    batch_size=batch_size,
    multi_view_mode=multi_view_mode, 
)

Data.insert(0, denoised)

# Optional: Wasser auf alle Datensätze addieren
if add_water:
    raw_data = np.load(data_dir / "data.npy")
    suppressed_water = np.load(data_dir / "SupressedWater.npy")
    water = raw_data - suppressed_water

    Data = [d + water for d in Data]

# auch fft
Data_ft = [np.fft.fftshift(np.fft.fft(d, axis=-2), axes=-2) for d in Data]

In [ ]:
# Data_denoised = np.swapaxes(Data[0], 0, 1)
# #Data_noisy = np.swapaxes(Data[1], 0, 1)

# sio.savemat(f'3TVol01_Denoised_3D_WaterSuppressed.mat', {'Data': Data_denoised})
# #sio.savemat(f'Vol{Vol}_Noisy_after_Walinet.mat', {'Data': Data_noisy})

# Optional als matlab datei speichern

In [ ]:
# import numpy as np 
# from scipy.io import loadmat, savemat

# ### CAREFUL DISTINGUISH BETWEEN LESION AND NO LESION

# noisy = np.load(f"/workspace/Denoising/datasets/DMI/Simulations/SNRExperiments_double_radius/SNR{SNR}/Noise_6/data.npy")

# #/Lade Par und mask aus der B0_1.mat-Datei
# b0_data = loadmat(f"/workspace/Denoising/datasets/DMI/Simulations/Lesion_double_radius/GT/Lesion_120pts_DoubleRadius.mat") 
# par = b0_data['csi_data_lr']['Par'][0, 0]
# mask = b0_data['csi_data_lr']['mask'][0, 0]

# # Deine Datensätze
# data_dicts = {
#     f'SNR{SNR}_6_deep_tmppca.mat': Data[0],
#     #f'SNR{SNR}_6_noisy.mat': noisy,
#     #f'SNR{SNR}_6_tmppca.mat': Data[1],
#     # f'SNR{SNR}_GT.mat': Data[2]
# }

# # Für jede Datei speichern
# for filename, data in data_dicts.items():
#     savemat(filename, {
#         'csi_data_lr': {
#             'Data': data,
#             'Par': par,
#             'mask': mask
#         }
#     })

# Compare spectral slice

In [ ]:
# fig, axes = plot_z_slices(
#     Data_ft,
#     Titles,
#     t=700,
#     T=7,
#     share_clim="global",
# )

# plt.show()

# Compare Spectra

In [ ]:
# _ = plot_voxel_spectra_over_z(Data_ft, Titles, x=15, y=15, min_t_index=0, mag=True, T = -1)#, z_max=30)#, min_t_index=500, max_t_index=600)

# plt.show()

# Compare average spectra
Here I compare the average spectrum over time (which is a high SNR estimate)

In [ ]:
Avg = [np.mean(d, axis=(0,1,2)) for d in Data_ft]

_ = plot_average_spectra_over_T(
    Avg,
    Titles,
    n_cols=2,
)
plt.show()

# Compare spectra interactively

In [ ]:
T = 4

view = 2
average = False

if average:
    data1 = np.imag(np.mean(Data_ft[0], axis=0))
else:
    data1 = np.imag(Data_ft[0][...,T])

data2 = np.imag(Data_ft[2][...,T])

image_volume = np.abs(np.mean(data1, axis=-1))#magnitude   # shape (X, Y, Z)

interactive_spectra_viewer(
    image_volume=image_volume,
    spectrum_func1=lambda x, y, z: data1[x, y, z, :],
    spectrum_func2=lambda x, y, z: data2[x, y, z, :],
    label1="Deep",
    label2="GT",
    f_min=0,
    #f_max=60,
    cmap="plasma",
)

In [ ]:
def plot_real_imag_ft(data_5d, x, y, z, title=None, save_path=None):
    """
    Plot real and imaginary parts of one voxel's f x T spectrum.

    Parameters
    ----------
    data_5d : np.ndarray
        Complex array with shape (x, y, z, f, T).
    x, y, z : int
        Voxel coordinates.
    title : str or None
        Optional title above the figure.
    save_path : str or None
        Optional path, e.g. "panel.pdf".
    """

    voxel = data_5d[x, y, z, ...]  # (f, T)

    real = np.real(voxel).T
    imag = np.imag(voxel).T

    data = np.concatenate([real.ravel(), imag.ravel()])
    vmin = np.percentile(data, 1)
    vmax = np.percentile(data, 99)

    fig, ax = plt.subplots(2, 1, figsize=(2.2, 2.0), dpi=300, sharex=True)

    for a, img, label in zip(ax, [real, imag], ["Real part", "Imaginary part"]):
        a.imshow(
            img,
            cmap="gray",
            vmin=vmin,
            vmax=vmax,
            origin="lower",
            aspect="auto",
            interpolation="nearest",
        )

        a.text(
            0.03, 0.88, label,
            transform=a.transAxes,
            fontsize=6,
            fontweight="bold",
            ha="left",
            va="top",
            color="white",
            bbox=dict(
                facecolor="black",
                alpha=0.4,
                edgecolor="none",
                pad=1.5
            )
        )
        
        a.set_xticks([])
        a.set_yticks([])

        for spine in a.spines.values():
            spine.set_linewidth(0.6)

    # gemeinsame Achsenlabels
    fig.text(0.02, 0.5, "Repetition", rotation=90,
             va="center", ha="center", fontsize=7)
    fig.text(0.5, 0.04, "Frequency",
             va="center", ha="center", fontsize=7)

    # optionaler Titel
    if title is not None:
        fig.suptitle(title, fontsize=8, y=0.99)

    plt.subplots_adjust(left=0.12, right=0.98,
                        top=0.90 if title else 0.98,
                        bottom=0.13, hspace=0.03)

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", pad_inches=0.02)

    return fig, ax

# Visualize Spectra evolving over time

In [ ]:
fig, ax = plot_real_imag_ft(Data_ft[1], 9, 10, 11, title="tMPPCA", save_path="SavedGraphics/tMPPCA.pdf")

# oder direkt speichern:
#fig, ax = plot_real_imag_ft(Data_ft[0], 9, 10, 11, save_path="real_imag_panel.pdf")